# Dropout and Normalization: Two Layers Every Diagram Has

The runnable code of [Dropout and Normalization: Two Layers Every Diagram Has](https://residual-stream.lvtan71.workers.dev/posts/dropout-and-normalization/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/dropout-and-normalization.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 7 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "dropout-and-normalization"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/dropout-and-normalization/net.py', 'shared/dropout-and-normalization/norms.py', 'shared/initialization-and-optimizers/optim.py', 'shared/softmax-and-cross-entropy/router4.py', 'shared/features-overfitting-regularization/router_data.py', 'shared/softmax-and-cross-entropy/softmax_basics.py', 'data/router-questions.csv']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## Dropout: training a crowd of thinner networks

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/dropout-and-normalization/#dropout-training-a-crowd-of-thinner-networks)

In [ ]:
import numpy as np
from softmax_basics import softmax, cross_entropy
from norms import layernorm, layernorm_back, rmsnorm, rmsnorm_back

def init(sizes, rng, std=lambda m: np.sqrt(2 / m), norm=None):
    layers = []
    for m, n in zip(sizes[:-1], sizes[1:]):
        L = {"W": rng.normal(0, std(m), (m, n)), "b": np.zeros(n)}
        if norm and len(layers) < len(sizes) - 2:           # hidden layers only
            L["g"] = np.ones(n)
            if norm == "layer":
                L["beta"] = np.zeros(n)
        layers.append(L)
    return layers

def forward(layers, X, norm=None, drop=0.0, rng=None):
    """Each hidden layer: linear, then (optionally) a norm, ReLU, then dropout."""
    h, caches = X, []
    for i, L in enumerate(layers):
        z = h @ L["W"] + L["b"]
        if i == len(layers) - 1:
            caches.append((h, None, None, None))
            return z, caches
        nc = None
        if norm == "layer":
            z, nc = layernorm(z, L["g"], L["beta"])
        elif norm == "rms":
            z, nc = rmsnorm(z, L["g"])
        a = np.maximum(z, 0)
        mask = None
        if drop and rng is not None:                       # inverted dropout, training only
            mask = (rng.random(a.shape) >= drop) / (1 - drop)
            a = a * mask
        caches.append((h, z, nc, mask))
        h = a

def loss_and_grads(layers, X, y, norm=None, drop=0.0, rng=None):
    logits, caches = forward(layers, X, norm, drop, rng)
    g = (softmax(logits) - np.eye(logits.shape[1])[y]) / len(y)      # p - y
    grads = [None] * len(layers)
    for i in reversed(range(len(layers))):
        h, z, nc, mask = caches[i]
        if i < len(layers) - 1:                            # back through dropout, ReLU, norm
            if mask is not None:
                g = g * mask
            g = g * (z > 0)
            G = {}
            if norm == "layer":
                g, G["g"], G["beta"] = layernorm_back(g, layers[i]["g"], nc)
            elif norm == "rms":
                g, G["g"] = rmsnorm_back(g, layers[i]["g"], nc)
        else:
            G = {}
        G.update({"W": h.T @ g, "b": g.sum(0)})
        grads[i] = G
        g = g @ layers[i]["W"].T
    return cross_entropy(logits, y), grads

def fit(layers, opt, X, y, steps, rng, norm=None, drop=0.0, batch=16, every=None):
    """Mini-batch training. Returns the full training loss at the steps in `every`."""
    seen = {}
    for t in range(1, steps + 1):
        idx = rng.choice(len(y), batch, replace=False)
        _, grads = loss_and_grads(layers, X[idx], y[idx], norm, drop, rng)
        for i, (L, G) in enumerate(zip(layers, grads)):
            for k in L:
                opt.step(L[k], G[k], (i, k))
        if every and t in every:
            seen[t] = cross_entropy(forward(layers, X, norm)[0], y)
    return seen

In [ ]:
import numpy as np
from net import init, loss_and_grads

# The whole network's backward pass, with each norm, against finite differences.
rng = np.random.default_rng(0)
X, y = rng.normal(size=(6, 5)), rng.integers(0, 4, 6)
for norm in (None, "layer", "rms"):
    layers, worst = init([5, 7, 7, 4], rng, norm=norm), 0.0
    _, grads = loss_and_grads(layers, X, y, norm)
    for L, G in zip(layers, grads):
        for k in L:
            for i in np.ndindex(L[k].shape):
                old = L[k][i]
                L[k][i] = old + 1e-6; up = loss_and_grads(layers, X, y, norm)[0]
                L[k][i] = old - 1e-6; down = loss_and_grads(layers, X, y, norm)[0]
                L[k][i] = old
                num = (up - down) / 2e-6
                worst = max(worst, abs(num - G[k][i]) / max(1e-12, abs(num) + abs(G[k][i])))
    print(f"norm={norm}: largest relative error {worst:.1e}")

The post printed:

```text
norm=None: largest relative error 3.8e-08
norm=layer: largest relative error 6.7e-08
norm=rms: largest relative error 4.9e-07
```

In [ ]:
%%writefile step3.py
import sys
import numpy as np
from router_data import split
from router4 import four_way
from net import init, fit, forward
from softmax_basics import softmax, cross_entropy
from optim import Adam

# A two-hidden-layer router that can memorize its 120 training questions.
# Does dropout help it, alone or with AdamW's weight decay ("Initialization and Optimizers")?
STEPS = int(sys.argv[1]) if len(sys.argv) > 1 else 3000
CONFIGS = [(p, lam) for lam in (0.0, 0.3, 1.0, 3.0, 10.0) for p in (0.0, 0.1, 0.3, 0.5, 0.7)]
rows = {c: [] for c in CONFIGS}
for seed in range(20):
    ((Xtr, ytr), (Xva, yva), (Xte, yte)), _ = four_way(*split(seed))
    for p, lam in CONFIGS:
        rng = np.random.default_rng(seed)
        layers = init([Xtr.shape[1], 64, 64, 4], rng)
        fit(layers, Adam(1e-3, decay=lam), Xtr, ytr, STEPS, rng, drop=p)
        zv, zt = forward(layers, Xva)[0], forward(layers, Xte)[0]      # no dropout at test time
        top2 = np.sort(zv, 1)[:, -2:]                              # the two largest logits
        wrong = zv.argmax(1) != yva
        rows[(p, lam)].append((cross_entropy(zv, yva), (zv.argmax(1) == yva).mean(),
                               softmax(zv).max(1).mean(), cross_entropy(zt, yte),
                               (top2[:, 1] - top2[:, 0]).mean(), softmax(zv).max(1)[wrong].mean()))
corrected = np.sqrt(1 / 20 + 40 / 120)
se = lambda d: f"{d.mean():+.3f} ± {d.std(ddof=1) * corrected:.3f}"
base = np.array(rows[(0.0, 0.0)])
print(f"{STEPS:,} steps of Adam. Differences are paired, against no dropout at the same decay.")
for (p, lam), r in rows.items():
    r, same = np.array(r), np.array(rows[(0.0, lam)])
    print(f"dropout {p:<4} decay {lam:<4} val CE {r[:, 0].mean():.3f} "
          f"{se(r[:, 0] - same[:, 0]) if p else '':>15s}  val acc {r[:, 1].mean():.3f} "
          f"{se(r[:, 1] - same[:, 1]) if p else '':>15s}  mean top p {r[:, 2].mean():.3f}  "
          f"top-2 logit gap {r[:, 4].mean():5.2f}  top p when wrong {r[:, 5].mean():.3f}")
pick = lambda cs: min(cs, key=lambda c: np.mean(np.array(rows[c])[:, 0]))   # on validation
best, decay_only = pick(CONFIGS), pick([c for c in CONFIGS if c[0] == 0.0])
t, d = np.array(rows[best])[:, 3], np.array(rows[decay_only])[:, 3]
print(f"chosen on validation: dropout {best[0]}, decay {best[1]}; without dropout: decay {decay_only[1]}")
print(f"test CE: {t.mean():.3f} chosen, {d.mean():.3f} decay only, {base[:, 3].mean():.3f} neither. "
      f"Dropout's own share: {se(t - d)}; decay's: {se(d - base[:, 3])}")

In [ ]:
%run step3.py

The post printed:

```text
3,000 steps of Adam. Differences are paired, against no dropout at the same decay.
dropout 0.0  decay 0.0  val CE 1.115                  val acc 0.667                  mean top p 0.857  top-2 logit gap  3.85  top p when wrong 0.752
dropout 0.1  decay 0.0  val CE 1.228  +0.113 ± 0.052  val acc 0.680  +0.013 ± 0.023  mean top p 0.874  top-2 logit gap  4.43  top p when wrong 0.786
dropout 0.3  decay 0.0  val CE 1.462  +0.347 ± 0.080  val acc 0.686  +0.019 ± 0.032  mean top p 0.910  top-2 logit gap  5.80  top p when wrong 0.840
dropout 0.5  decay 0.0  val CE 1.661  +0.546 ± 0.132  val acc 0.679  +0.011 ± 0.027  mean top p 0.926  top-2 logit gap  6.93  top p when wrong 0.860
dropout 0.7  decay 0.0  val CE 1.513  +0.397 ± 0.116  val acc 0.684  +0.016 ± 0.029  mean top p 0.913  top-2 logit gap  6.54  top p when wrong 0.843
dropout 0.0  decay 0.3  val CE 0.953                  val acc 0.683                  mean top p 0.807  top-2 logit gap  2.95  top p when wrong 0.711
dropout 0.1  decay 0.3  val CE 0.950  -0.003 ± 0.034  val acc 0.693  +0.010 ± 0.027  mean top p 0.822  top-2 logit gap  3.18  top p when wrong 0.719
dropout 0.3  decay 0.3  val CE 0.974  +0.021 ± 0.044  val acc 0.704  +0.021 ± 0.038  mean top p 0.845  top-2 logit gap  3.58  top p when wrong 0.751
dropout 0.5  decay 0.3  val CE 1.018  +0.065 ± 0.058  val acc 0.704  +0.021 ± 0.040  mean top p 0.864  top-2 logit gap  3.99  top p when wrong 0.772
dropout 0.7  decay 0.3  val CE 1.016  +0.062 ± 0.060  val acc 0.701  +0.019 ± 0.034  mean top p 0.864  top-2 logit gap  4.12  top p when wrong 0.767
dropout 0.0  decay 1.0  val CE 0.879                  val acc 0.680                  mean top p 0.762  top-2 logit gap  2.41  top p when wrong 0.658
dropout 0.1  decay 1.0  val CE 0.869  -0.011 ± 0.046  val acc 0.689  +0.009 ± 0.024  mean top p 0.787  top-2 logit gap  2.67  top p when wrong 0.675
dropout 0.3  decay 1.0  val CE 0.865  -0.014 ± 0.054  val acc 0.698  +0.017 ± 0.041  mean top p 0.792  top-2 logit gap  2.76  top p when wrong 0.683
dropout 0.5  decay 1.0  val CE 0.846  -0.033 ± 0.057  val acc 0.715  +0.035 ± 0.034  mean top p 0.793  top-2 logit gap  2.76  top p when wrong 0.692
dropout 0.7  decay 1.0  val CE 0.809  -0.070 ± 0.056  val acc 0.714  +0.034 ± 0.032  mean top p 0.775  top-2 logit gap  2.55  top p when wrong 0.662
dropout 0.0  decay 3.0  val CE 0.809                  val acc 0.694                  mean top p 0.719  top-2 logit gap  1.94  top p when wrong 0.605
dropout 0.1  decay 3.0  val CE 0.806  -0.003 ± 0.017  val acc 0.690  -0.004 ± 0.020  mean top p 0.715  top-2 logit gap  1.93  top p when wrong 0.596
dropout 0.3  decay 3.0  val CE 0.799  -0.010 ± 0.022  val acc 0.689  -0.005 ± 0.024  mean top p 0.705  top-2 logit gap  1.86  top p when wrong 0.582
dropout 0.5  decay 3.0  val CE 0.781  -0.028 ± 0.025  val acc 0.707  +0.014 ± 0.025  mean top p 0.678  top-2 logit gap  1.68  top p when wrong 0.561
dropout 0.7  decay 3.0  val CE 0.806  -0.004 ± 0.035  val acc 0.698  +0.004 ± 0.025  mean top p 0.606  top-2 logit gap  1.25  top p when wrong 0.492
dropout 0.0  decay 10.0 val CE 0.940                  val acc 0.670                  mean top p 0.482  top-2 logit gap  0.67  top p when wrong 0.411
dropout 0.1  decay 10.0 val CE 0.962  +0.022 ± 0.018  val acc 0.647  -0.022 ± 0.028  mean top p 0.469  top-2 logit gap  0.62  top p when wrong 0.403
dropout 0.3  decay 10.0 val CE 1.021  +0.081 ± 0.027  val acc 0.629  -0.041 ± 0.043  mean top p 0.430  top-2 logit gap  0.46  top p when wrong 0.376
dropout 0.5  decay 10.0 val CE 1.104  +0.164 ± 0.031  val acc 0.620  -0.050 ± 0.042  mean top p 0.380  top-2 logit gap  0.33  top p when wrong 0.342
dropout 0.7  decay 10.0 val CE 1.246  +0.306 ± 0.036  val acc 0.443  -0.227 ± 0.042  mean top p 0.312  top-2 logit gap  0.07  top p when wrong 0.304
chosen on validation: dropout 0.5, decay 3.0; without dropout: decay 3.0
test CE: 0.776 chosen, 0.798 decay only, 1.083 neither. Dropout's own share: -0.022 ± 0.024; decay's: -0.285 ± 0.097
```

## BatchNorm, and why language models don't use it

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/dropout-and-normalization/#batchnorm-and-why-language-models-dont-use-it)

In [ ]:
import numpy as np
from router_data import english, split
from router4 import four_way
from norms import batchnorm, layernorm

rng = np.random.default_rng(0)
tr, va, te = split(0)
((X, y), (Xall, _)), _ = four_way(tr, english)
W = rng.normal(0, np.sqrt(2 / X.shape[1]), (X.shape[1], 64))   # one He-initialized layer
Z = Xall @ W                                                     # its pre-activations, all 200 questions
g, b = np.ones(64), np.zeros(64)

# 1. The same question, normalized inside two different batches of 16.
others = rng.permutation(np.arange(1, 200))
out1 = batchnorm(Z[np.r_[0, others[:15]]], g, b)[0][0]
out2 = batchnorm(Z[np.r_[0, others[15:30]]], g, b)[0][0]
print(f"one question, two batches: its BatchNorm outputs differ by {np.abs(out1 - out2).mean():.2f} "
      f"on average (their typical size is {np.abs(out1).mean():.2f})")

# 2. How far a batch's mean is from the mean over all 200 questions, by batch size.
full_mu, full_sd = Z.mean(0), Z.std(0)
for B in (2, 8, 32, 128):
    gaps = [np.abs(Z[rng.choice(200, B, replace=False)].mean(0) - full_mu).mean() / full_sd.mean()
            for _ in range(500)]
    print(f"batch of {B:3d}: batch mean is off by {np.mean(gaps):.2f} standard deviations on average")

# 3. At inference, a batch of one: every feature equals its own mean.
print(f"batch of 1: BatchNorm output {np.abs(batchnorm(Z[:1], g, b)[0]).max():.1f} everywhere; "
      f"LayerNorm output still varies, size {np.abs(layernorm(Z[:1], g, b)[0]).mean():.2f}")

# 4. A sequence: normalize 8 token vectors together (over positions), then change
# only the *last* token. Does the *first* token's output change?
seq = rng.normal(size=(8, 64))
later = seq.copy(); later[-1] += 3.0
bn_gap = np.abs(batchnorm(seq, g, b)[0][0] - batchnorm(later, g, b)[0][0]).max()
ln_gap = np.abs(layernorm(seq, g, b)[0][0] - layernorm(later, g, b)[0][0]).max()
print(f"change the last token only: first token's output moves by {bn_gap:.3f} with BatchNorm, "
      f"{ln_gap:.3f} with LayerNorm")

The post printed:

```text
one question, two batches: its BatchNorm outputs differ by 0.37 on average (their typical size is 0.91)
batch of   2: batch mean is off by 0.55 standard deviations on average
batch of   8: batch mean is off by 0.28 standard deviations on average
batch of  32: batch mean is off by 0.13 standard deviations on average
batch of 128: batch mean is off by 0.04 standard deviations on average
batch of 1: BatchNorm output 0.0 everywhere; LayerNorm output still varies, size 0.77
change the last token only: first token's output moves by 1.294 with BatchNorm, 0.000 with LayerNorm
```

## LayerNorm: normalize each example by itself

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/dropout-and-normalization/#layernorm-normalize-each-example-by-itself)

In [ ]:
import numpy as np

EPS = 1e-5

def layernorm(x, g, b):
    """Normalize each row to mean 0, variance 1, then scale by g and shift by b."""
    mu = x.mean(-1, keepdims=True)
    var = x.var(-1, keepdims=True)
    xhat = (x - mu) / np.sqrt(var + EPS)
    return xhat * g + b, (xhat, var)

def layernorm_back(dy, g, cache):
    xhat, var = cache
    dxhat = dy * g
    # The gradient of a normalization removes the parts of dxhat that would
    # only move the mean or the overall scale, which the normalization undoes.
    dx = (dxhat - dxhat.mean(-1, keepdims=True)
          - xhat * (dxhat * xhat).mean(-1, keepdims=True)) / np.sqrt(var + EPS)
    return dx, (dy * xhat).sum(0), dy.sum(0)            # dx, dg, db

def rmsnorm(x, g):
    """Divide each row by its root mean square, then scale by g. No mean, no shift."""
    rms = np.sqrt((x ** 2).mean(-1, keepdims=True) + EPS)
    xhat = x / rms
    return xhat * g, (xhat, rms)

def rmsnorm_back(dy, g, cache):
    xhat, rms = cache
    dxhat = dy * g
    dx = (dxhat - xhat * (dxhat * xhat).mean(-1, keepdims=True)) / rms
    return dx, (dy * xhat).sum(0)                        # dx, dg

def batchnorm(x, g, b):
    """Normalize each *column* over the batch: every example depends on the others."""
    mu, var = x.mean(0), x.var(0)
    xhat = (x - mu) / np.sqrt(var + EPS)
    return xhat * g + b, (mu, var)

if __name__ == "__main__":
    # A worked vector, then the same vector shifted by 100.
    g, b = np.ones(4), np.zeros(4)
    for x in (np.array([[2.0, 4.0, 6.0, 8.0]]), np.array([[102.0, 104.0, 106.0, 108.0]])):
        print(f"x = {x[0]}:  LayerNorm {np.round(layernorm(x, g, b)[0][0], 3)}   "
              f"RMSNorm {np.round(rmsnorm(x, g)[0][0], 3)}")
    # Check both backward passes against finite differences, as in "Backpropagation, by Hand".
    rng = np.random.default_rng(0)
    x, g, b, dy = rng.normal(size=(3, 5)), rng.normal(size=5), rng.normal(size=5), rng.normal(size=(3, 5))
    for name, f, back in [("LayerNorm", lambda x: layernorm(x, g, b), lambda c: layernorm_back(dy, g, c)[0]),
                          ("RMSNorm", lambda x: rmsnorm(x, g), lambda c: rmsnorm_back(dy, g, c)[0])]:
        analytic, worst = back(f(x)[1]), 0.0
        for i in np.ndindex(x.shape):
            e = np.zeros_like(x); e[i] = 1e-6
            numeric = ((f(x + e)[0] - f(x - e)[0]) * dy).sum() / 2e-6
            worst = max(worst, abs(numeric - analytic[i]) / max(1e-12, abs(numeric) + abs(analytic[i])))
        print(f"{name} backward vs finite differences: largest relative error {worst:.1e}")

The post printed:

```text
x = [2. 4. 6. 8.]:  LayerNorm [-1.342 -0.447  0.447  1.342]   RMSNorm [0.365 0.73  1.095 1.461]
x = [102. 104. 106. 108.]:  LayerNorm [-1.342 -0.447  0.447  1.342]   RMSNorm [0.971 0.99  1.009 1.028]
LayerNorm backward vs finite differences: largest relative error 1.9e-09
RMSNorm backward vs finite differences: largest relative error 2.4e-10
```

## RMSNorm: drop the mean

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/dropout-and-normalization/#rmsnorm-drop-the-mean)

In [ ]:
# One normalization over a training batch of 8 x 8,192 tokens at width 4,096,
# in BF16 on an H100 SXM, against one of the layer's matmuls.
tokens, D, byte = 8 * 8192, 4096, 2
vector_flops = 67e12        # FP32 on the ordinary cores, where a norm's math runs
tensor_flops = 989e12       # BF16 on the tensor cores: matmuls only
bandwidth = 3.35e12
x_bytes = tokens * D * byte                           # one full read (or write) of the activations
for name, flops_per_elem, passes_unfused in [("LayerNorm", 8, 4), ("RMSNorm", 5, 3)]:
    t_math = tokens * D * flops_per_elem / vector_flops
    print(f"{name:9s} math {t_math * 1e6:5.0f} us   memory: unfused {passes_unfused} passes "
          f"{passes_unfused * x_bytes / bandwidth * 1e3:.2f} ms, fused 2 passes {2 * x_bytes / bandwidth * 1e3:.2f} ms")
matmul = 2 * tokens * D * D / tensor_flops
print(f"one {D} x {D} matmul on the same tokens, at tensor-core peak: {matmul * 1e3:.2f} ms")

The post printed:

```text
LayerNorm math    32 us   memory: unfused 4 passes 0.64 ms, fused 2 passes 0.32 ms
RMSNorm   math    20 us   memory: unfused 3 passes 0.48 ms, fused 2 passes 0.32 ms
one 4096 x 4096 matmul on the same tokens, at tensor-core peak: 2.22 ms
```

## What normalization does to training

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/dropout-and-normalization/#what-normalization-does-to-training)

In [ ]:
import numpy as np
from norms import layernorm, layernorm_back

def signal(std, depth=20, width=256, seed=0):
    """The 20-layer test from "Initialization and Optimizers", with a LayerNorm before every ReLU."""
    rng = np.random.default_rng(seed)
    h = rng.normal(size=(512, width))
    Ws = [rng.normal(0, std, (width, width)) for _ in range(depth)]
    g, b = np.ones(width), np.zeros(width)
    caches, hs = [], []
    for W in Ws:
        hs.append(h)
        z, c = layernorm(h @ W, g, b)
        caches.append((z, c))
        h = np.maximum(z, 0)
    top = np.sqrt((h ** 2).mean())
    d = rng.normal(size=h.shape)
    wgrad = []
    for W, (z, c), hin in zip(Ws[::-1], caches[::-1], hs[::-1]):
        d = layernorm_back(d * (z > 0), g, c)[0]
        wgrad.append(np.sqrt(((hin.T @ d) ** 2).mean()))
        d = d @ W.T
    return top, np.sqrt((d ** 2).mean()), wgrad[-1]

print(f"{'weight std':>10s}  {'activation size, layer 20':>26s}  {'gradient reaching layer 1':>26s}  "
      f"{'gradient on W, layer 1':>23s}")
for std in (0.01, np.sqrt(2 / 256), 1.0):
    a, gx, gw = np.median([signal(std, seed=s) for s in range(5)], axis=0)
    print(f"{std:10.3f}  {a:26.2f}  {gx:26.2e}  {gw:23.2e}")

The post printed:

```text
weight std   activation size, layer 20   gradient reaching layer 1   gradient on W, layer 1
     0.010                        0.71                    7.84e-01                 1.11e+02
     0.088                        0.71                    7.85e-01                 1.25e+01
     1.000                        0.71                    7.85e-01                 1.11e+00
```

In [ ]:
import numpy as np
from router_data import split
from router4 import four_way
from net import init, fit
from optim import SGD, Adam

# The ten-layer router of "Initialization and Optimizers", from its fatal start (std 0.01) and from He,
# with and without a normalization layer in every hidden layer. For each,
# sweep the learning rate and keep the best: can *any* step size train it?
grids = {"SGD": (SGD, [1e-4, 1e-3, 1e-2, 1e-1, 1.0]), "Adam": (Adam, [1e-5, 1e-4, 1e-3, 1e-2])}
out = {}
for seed in range(5):
    ((X, y), _, _), _ = four_way(*split(seed))
    sizes = [X.shape[1]] + [64] * 10 + [4]
    for start, std in [("std 0.01", lambda m: 0.01), ("He", lambda m: np.sqrt(2 / m))]:
        for norm in (None, "layer", "rms"):
            for opt_name, (make, etas) in grids.items():
                for eta in etas:
                    rng = np.random.default_rng(seed)
                    layers = init(sizes, rng, std, norm)
                    with np.errstate(all="ignore"):
                        seen = fit(layers, make(eta), X, y, 1000, rng, norm, every={1000})
                    out.setdefault((start, norm or "none", opt_name, eta), []).append(seen[1000])
print(f"{'start':9s} {'norm':6s} {'optimizer':5s}  train loss after 1,000 steps, by learning rate")
for (start, norm, opt_name), (make, etas) in [((s, n, o), grids[o]) for s in ("std 0.01", "He")
                                              for n in ("none", "layer", "rms") for o in grids]:
    cells = [f"{e:g}: {np.nan_to_num(np.median(out[(start, norm, opt_name, e)]), nan=np.inf):.3f}" for e in etas]
    print(f"{start:9s} {norm:6s} {opt_name:5s}  " + "  ".join(cells))

The post printed:

```text
start     norm   optimizer  train loss after 1,000 steps, by learning rate
std 0.01  none   SGD    0.0001: 1.386  0.001: 1.386  0.01: 1.386  0.1: 1.387  1: 1.392
std 0.01  none   Adam   1e-05: 1.386  0.0001: 1.386  0.001: 1.386  0.01: 1.387
std 0.01  layer  SGD    0.0001: 0.872  0.001: 0.069  0.01: 1.387  0.1: 1.387  1: 1.392
std 0.01  layer  Adam   1e-05: 0.808  0.0001: 0.072  0.001: 1.386  0.01: 1.387
std 0.01  rms    SGD    0.0001: 0.798  0.001: 0.062  0.01: 1.386  0.1: 1.387  1: 1.392
std 0.01  rms    Adam   1e-05: 0.722  0.0001: 0.055  0.001: 1.196  0.01: 1.387
He        none   SGD    0.0001: 1.385  0.001: 1.286  0.01: 0.002  0.1: 0.000  1: inf
He        none   Adam   1e-05: 1.230  0.0001: 0.001  0.001: 0.000  0.01: 0.000
He        layer  SGD    0.0001: 1.214  0.001: 0.226  0.01: 0.006  0.1: 0.001  1: 1.392
He        layer  Adam   1e-05: 0.803  0.0001: 0.010  0.001: 0.000  0.01: 0.506
He        rms    SGD    0.0001: 1.221  0.001: 0.207  0.01: 0.005  0.1: 0.001  1: 1.392
He        rms    Adam   1e-05: 0.810  0.0001: 0.009  0.001: 0.000  0.01: 0.000
```